# Lab 1 — Pandas: transformations, aggregations and joins

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np

print(f"Pandas version: {pd.__version__}")
print(f"NumPy version:  {np.__version__}")

In [ ]:
# --- Creating from a dictionary --- JSON API, configuration files, etc.
data = {
    "product":    ["Notebook,laptop", "Mouse", "Keyboard", "Monitor", "Headset",
                   "Webcam", "External HD", "Pendrive", "USB Cable", "USB Hub"],
    "category":   ["Computers", "Peripherals", "Peripherals", "Computers", "Audio",
                   "Peripherals", "Storage", "Storage", "Cables", "Peripherals"],
    "price":      [3500, 85, 150, 1200, 250, 320, 400, 60, 25, 120],
    "stock":      [10, 45, 30, 8, 20, 15, 12, 100, 200, 18],
    "rating":     [4.8, 4.2, 4.5, 4.7, 3.9, 4.1, 4.6, 4.0, None, 4.3]
}

df = pd.DataFrame(data)
df

---
## 2. Transformations and Cleaning
### 2.1 Creating and modifying columns

In [ ]:
# New column — total stock value
df["stock_value"] = df["price"] * df["stock"]

# Column with a conditional operation (np.where = IF in SQL)
df["expensive"] = np.where(df["price"] > 500, "Yes", "No")

# Text column
df["product_upper"] = df["product"].str.upper() # lower()

df[["product", "price", "stock", "stock_value", "expensive", "product_upper"]].head(5)

### 2.2 `.apply()`: applying functions to columns/rows
> In Spark this is the equivalent of UDFs (User Defined Functions)

In [ ]:
# Applying a function to the whole column
def classify_price(price):
    if price < 100:
        return "Budget"
    elif price < 500:
        return "Mid-range"
    return "Premium"

df["price_tier"] = df["price"].apply(classify_price)

# With lambda — same thing in one line
df["price_with_shipping"] = df["price"].apply(lambda x: x + 15 if x < 200 else x)

df[["product", "price", "price_tier", "price_with_shipping"]]

In [ ]:
# apply across multiple columns (axis=1 = row by row)
df["summary"] = df.apply(
    lambda row: f"{row['product']} ({row['category']}) — R$ {row['price']:.2f}",
    axis=1
)
df["summary"].head(4)

### 2.3 Handling null values

In [ ]:
df.info()

In [ ]:
print("Before — nulls in 'rating':", df["rating"].isnull().sum())

# Option 1: Fill with a fixed value
df_ = df.copy()
df_["rating"] = df_["rating"].fillna(0)

# print("After — nulls in 'rating':", df["rating"].isnull().sum())
df_[["product", "rating"]]

In [ ]:
df = df.copy()

# Option 2: Fill with the mean (more realistic)
mean_rating = df["rating"].mean()
df["rating"] = df["rating"].fillna(mean_rating)

# Option 3: Drop rows with nulls
# df_clean = df.dropna(subset=["rating"])

# print("After — nulls in 'rating':", df["rating"].isnull().sum())
df[["product", "rating"]]

### 2.4 Data types and conversion

In [ ]:
print("Current types:")
print(df.dtypes)

# Converting types
df["price"] = df["price"].astype(float)
df["category"] = df["category"].astype("category")  # saves memory

print("\nTypes after conversion:")
print(df[["price", "category"]].dtypes)

### 2.5 Renaming and dropping columns

In [ ]:
# Rename
df = df.rename(columns={"price": "price_brl", "rating": "score"})

# Drop the helper columns we created
df = df.drop(columns=["product_upper", "summary", "expensive"])

print("Current columns:", df.columns.tolist())

---
## 3. Aggregations and GroupBy
> This concept is **identical** in Spark: `df.groupBy(...).agg(...)`

In [ ]:
df.head(10)

In [ ]:
# Simple aggregations per group
df.groupby("category")["price_brl"].mean().round(2)

In [ ]:
# Multiple metrics with .agg()
summary = df.groupby("category").agg(
    n_products   = ("product",    "count"),
    avg_price    = ("price_brl",  "mean"),
    max_price      = ("price_brl",  "max"),
    total_stock  = ("stock",    "sum"),
    avg_score     = ("score",       "mean")
).round(2).reset_index()

summary

In [ ]:
# Sorting the summary
summary.sort_values("n_products", ascending=False)

In [ ]:
# GroupBy + transform → keeps the original index (useful for creating columns)
df["category_avg_price"] = df.groupby("category")["price_brl"].transform("mean")
df["above_average"] = df["price_brl"] > df["category_avg_price"]

df[["product", "category", "price_brl", "category_avg_price", "above_average"]]

In [ ]:
# Pivot Table — matrix view (rows × columns)
# (Needs data with more variation; this example is illustrative)
df["tier"] = pd.cut(df["price_brl"],
                     bins=[0, 100, 500, 5000],
                     labels=["Cheap", "Medium", "Expensive"])

df

In [ ]:
pd.pivot_table(df,
               values="stock", # Column to aggregate
               index=["category"], # pivot: columns to group by (reference)
               columns="tier", # use as column headers
               aggfunc="sum",
               fill_value=0)

---
## 4. Joins and Combinations
> In Spark: `df1.join(df2, on="key", how="left")`

In [ ]:
# Creating a second DataFrame of suppliers
suppliers = pd.DataFrame({
    "category": ["Computers", "Peripherals", "Audio", "Storage"],
    "supplier": ["TechPro", "OfficeMax", "SoundWave", "DataStore"],
    "delivery_days": [7, 3, 5, 4]
})

suppliers

In [ ]:
df

In [ ]:
# INNER JOIN — matches only
df_join = df.merge(suppliers, on=["category"], how="left")
df_join[["product", "category", "price_brl", "supplier", "delivery_days"]]

In [ ]:
# INNER JOIN — matches only
df_join = df.merge(suppliers, on=["category"], how="right")
df_join[["product", "category", "price_brl", "supplier", "delivery_days"]]

In [ ]:
# INNER JOIN — matches only
df_join = df.merge(suppliers, on=["category"], how="inner")
df_join[["product", "category", "price_brl", "supplier", "delivery_days"]]

In [ ]:
# Available join types:
# how="inner"  → intersection (default)
# how="left"   → everything from the left + matches from the right (NaN if none)
# how="right"  → everything from the right + matches from the left
# how="outer"  → full union

# Example: LEFT JOIN
df_left = df.merge(suppliers, on="category", how="left")
print("Shape after left join:", df_left.shape)
df_left[["product", "category", "supplier"]].head()

In [ ]:
# pd.concat — stacking DataFrames (UNION in SQL)
df_computers = df[df["category"] == "Computers"].copy()
df_peripherals = df[df["category"] == "Peripherals"].copy()

df_combined = pd.concat([df_computers, df_peripherals], ignore_index=True)
print("Combined shape:", df_combined.shape)
df_combined[["product", "category"]]

In [ ]:
pd.concat([df_computers[["product", "category"]], df_peripherals[["product", "category"]]], ignore_index=True, axis=1)

---
## 5. Pipeline: Putting It All Together
> **Chaining operations** (`method chaining`) makes code cleaner and easier to read.  
> This style of writing is **identical** to the one used in PySpark!

In [ ]:
# Recreating the base DataFrame
data = {
    "product":    ["Notebook", "Mouse", "Keyboard", "Monitor", "Headset",
                   "Webcam", "External HD", "Pendrive", "USB Cable", "USB Hub"],
    "category":   ["Computers", "Peripherals", "Peripherals", "Computers", "Audio",
                   "Peripherals", "Storage", "Storage", "Cables", "Peripherals"],
    "price":      [3500, 85, 150, 1200, 250, 320, 400, 60, 25, 120],
    "stock":      [10, 45, 30, 8, 20, 15, 12, 100, 200, 18],
    "rating":     [4.8, 4.2, 4.5, 4.7, 3.9, 4.1, 4.6, 4.0, None, 4.3]
}
df_raw = pd.DataFrame(data)
df_raw

In [ ]:
df_ = df_raw.fillna({"rating": df_raw["rating"].mean()}).copy()
df_ = df_.assign(
        stock_value = lambda d: d["price"] * d["stock"],
        price_tier   = lambda d: pd.cut(d["price"],
                                         bins=[0, 100, 500, 5000],
                                         labels=["Cheap", "Medium", "Expensive"])
    ).copy()
df_ = df_.query("rating >= 4.0").copy()
df_ = df_[["product", "category", "price", "stock", "stock_value", "rating", "price_tier"]].copy()
df_.sort_values("stock_value", ascending=False, inplace=True)
df_.reset_index(drop=True,inplace=True)
df_

In [ ]:
# Full pipeline with method chaining
result = (
    df_raw
    # 1. Handle nulls
    .fillna({"rating": df_raw["rating"].mean()})

    # 2. Create new columns
    .assign(
        stock_value = lambda d: d["price"] * d["stock"],
        price_tier   = lambda d: pd.cut(d["price"],
                                         bins=[0, 100, 500, 5000],
                                         labels=["Cheap", "Medium", "Expensive"])
    )

    # 3. Filter
    .query("rating >= 4.0")

    # 4. Select the relevant columns
    [["product", "category", "price", "stock", "stock_value", "rating", "price_tier"]]

    # 5. Sort
    .sort_values("stock_value", ascending=False)

    # 6. Reset the index
    .reset_index(drop=True)
)

result

In [ ]:
# Final pipeline aggregation
(
    result
    .groupby(["category", "price_tier"])
    .agg(
        n            = ("product",        "count"),
        total_value    = ("stock_value",  "sum"),
        avg_score     = ("rating",      "mean")
    )
    .round(2)
    .dropna()
    .reset_index()
    .sort_values("total_value", ascending=False)
)

---
## Comparison Summary: Pandas → Spark

| Concept | Pandas | PySpark |
|----------|--------|---------|
| Create a DF | `pd.DataFrame(data)` | `spark.createDataFrame(data)` |
| Read CSV | `pd.read_csv("file.csv")` | `spark.read.csv("file.csv", header=True)` |
| Select columns | `df[["col1", "col2"]]` | `df.select("col1", "col2")` |
| Filter | `df[df["price"] > 100]` | `df.filter(df.price > 100)` |
| New column | `df["col"] = ...` | `df.withColumn("col", ...)` |
| GroupBy | `df.groupby("cat").agg(...)` | `df.groupBy("cat").agg(...)` |
| Join | `df.merge(df2, on="k", how="left")` | `df.join(df2, on="k", how="left")` |
| Sort | `df.sort_values("col")` | `df.orderBy("col")` |
| Count rows | `len(df)` | `df.count()` |
| Show | `df.head(5)` | `df.show(5)` |

>  **The concepts are the same; only the syntax changes a little!**

---

### Conceptual equivalence
| What we learned today | What comes with Spark |
|-----------------------|--------------------|
| `pd.DataFrame` | `spark.DataFrame` |
| Immediate execution | **Lazy evaluation** (transform → action) |
| 1 machine | Cluster of machines |
| `.apply()` / lambda | UDFs / Pandas UDFs |
| `.read_csv()` | `.read.parquet()` / `.read.delta()` |